# Decision Tree Classifier for UCI Letter Recognition

**Individual model:** Decision Tree  
**Task:** Predict one of 26 uppercase English letters (A–Z) from 16 numerical, image-derived features.

This notebook follows the Final Evaluation requirements: model suitability, implementation details, multiple model varieties, hyperparameter tuning, stratified validation, evaluation metrics, comparison, conclusion, limitations, and saved outputs.

**Research question:** How well can a Decision Tree classify the UCI Letter Recognition dataset, and which tested tree configuration gives the best cross-validation macro F1?


## 1. Libraries and reproducibility

`pandas` and `numpy` manage the data. `scikit-learn` provides the Decision Tree, preprocessing, cross-validation, GridSearchCV and evaluation metrics. `matplotlib` visualizes the confusion matrix, feature importance and a readable top portion of the final tree.

`RANDOM_STATE = 42` makes the data split and tree training reproducible.


In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn

from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree

RANDOM_STATE = 42

print("scikit-learn version:", sklearn.__version__)


## 2. Dataset and problem

The project uses the **UCI Letter Recognition** dataset. It contains 20,000 samples, 16 numerical features describing letter characteristics, and a target with 26 uppercase classes (A–Z).

The notebook first checks for the repository's local raw data file. In a fresh Colab runtime it can load the UCI CSV from the URL instead. The target is named `letter` in this notebook.


In [ ]:
UCI_CSV = "https://archive.ics.uci.edu/static/public/59/data.csv"

columns = [
    "letter", "x-box", "y-box", "width", "high", "onpix",
    "x-bar", "y-bar", "x2bar", "y2bar", "xybar", "x2ybr",
    "xy2br", "x-ege", "xegvy", "y-ege", "yegvx"
]

local_candidates = [
    Path.cwd() / "data" / "raw" / "letter-recognition.data",
    Path.cwd().parent / "data" / "raw" / "letter-recognition.data",
]

local_file = next((p for p in local_candidates if p.exists()), None)

if local_file is not None:
    raw = pd.read_csv(local_file, header=None, names=columns)
    print("Source:", local_file)
else:
    raw = pd.read_csv(UCI_CSV)
    # UCI's current CSV normally already has the correct column names.
    if raw.shape[1] == 17 and "letter" not in raw.columns and "lettr" not in raw.columns:
        raw.columns = columns
    if "lettr" in raw.columns:
        raw = raw.rename(columns={"lettr": "letter"})
    print("Source: UCI repository")

print("Shape:", raw.shape)
print("Missing values:", int(raw.isna().sum().sum()))
print("Number of classes:", raw["letter"].nunique())
display(raw.head())


## 3. Cleaning, encoding and hold-out split

Exact duplicate rows are removed. The target letters A–Z are encoded as integers 0–25. A **stratified 80/20 train-test split** is used so all 26 letter classes keep approximately the same class proportions.

The held-out test set is not used for hyperparameter selection. Model selection is performed only on the training partition using stratified cross-validation.


In [ ]:
data = raw.drop_duplicates().reset_index(drop=True)

encoder = LabelEncoder()
y = encoder.fit_transform(data["letter"])
X = data.drop(columns="letter")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Duplicates removed:", len(raw) - len(data))
print("Train:", X_train.shape)
print("Test:", X_test.shape)
print("Classes in train/test:", len(np.unique(y_train)), len(np.unique(y_test)))
print("First labels:", list(zip(encoder.classes_[:5], range(5))))


## 4. Why Decision Tree is suitable

A Decision Tree is suitable because this is a **supervised multi-class classification** problem with 16 numerical input features and 26 target classes. A tree can learn nonlinear decision rules and interactions between the letter measurements without requiring feature scaling.

It is also interpretable: we can inspect split rules and feature importances. However, a very deep tree can overfit, so parameters such as `max_depth`, `min_samples_split`, `min_samples_leaf`, `criterion`, and `ccp_alpha` should be tuned.


## 5. Baseline Decision Tree

The baseline uses `criterion="gini"` with default unrestricted depth and a fixed random state. This gives us a reference before tuning.

For Decision Trees, scaling is generally unnecessary because the algorithm splits features by thresholds rather than distances or gradient magnitudes.


In [ ]:
baseline = DecisionTreeClassifier(
    criterion="gini",
    random_state=RANDOM_STATE,
)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

baseline_scores = cross_val_score(
    baseline,
    X_train,
    y_train,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
)

print("Baseline fold macro F1:", baseline_scores)
print("Baseline mean macro F1:", baseline_scores.mean())
print("Baseline standard deviation:", baseline_scores.std())


## 6. Planned Decision Tree varieties

The tuning stage compares different Decision Tree varieties using:

- **Preprocessing:** all 16 original features, or mutual-information selection of the best 12 features.
- **Criterion:** `gini` and `entropy`.
- **Maximum depth:** 10, 20, and unrestricted (`None`).
- **Minimum samples per leaf:** 1 and 2.
- **Cost-complexity pruning (`ccp_alpha`):** 0.0 and 0.0005.

This creates **48 candidate configurations** (2 × 2 × 3 × 2 × 2). With 5-fold cross-validation, GridSearchCV performs 240 validation fits.

The purpose is to compare simpler/pruned trees with more flexible trees and determine which generalizes best.


In [ ]:
# Preprocessing branches.
# "passthrough" keeps all original features.
# SelectKBest is fitted inside each CV fold, preventing validation leakage.
preprocessors = [
    "passthrough",
    SelectKBest(score_func=mutual_info_classif, k=12),
]

search_pipeline = Pipeline([
    ("prep", "passthrough"),
    ("tree", DecisionTreeClassifier(random_state=RANDOM_STATE)),
])

parameter_grid = {
    "prep": preprocessors,
    "tree__criterion": ["gini", "entropy"],
    "tree__max_depth": [10, 20, None],
    "tree__min_samples_leaf": [1, 2],
    "tree__ccp_alpha": [0.0, 0.0005],
}

candidate_count = (
    len(preprocessors)
    * len(parameter_grid["tree__criterion"])
    * len(parameter_grid["tree__max_depth"])
    * len(parameter_grid["tree__min_samples_leaf"])
    * len(parameter_grid["tree__ccp_alpha"])
)

print("Candidate configurations:", candidate_count)
print("CV fits:", candidate_count * cv.get_n_splits())


## 7. Hyperparameter tuning with GridSearchCV

`GridSearchCV` tests every planned configuration using the same 5 stratified folds. The selection metric is **macro F1**, which gives equal importance to each of the 26 letters and balances precision and recall.

`refit=True` automatically retrains the best complete pipeline on the full training partition after model selection.


In [ ]:
search = GridSearchCV(
    estimator=search_pipeline,
    param_grid=parameter_grid,
    scoring="f1_macro",
    cv=cv,
    refit=True,
    n_jobs=-1,
    verbose=1,
    return_train_score=True,
)

search.fit(X_train, y_train)

print("Best CV macro F1:", search.best_score_)
print("Best settings:")
for key, value in search.best_params_.items():
    print(f"  {key}: {value}")


## 8. Compare all trained varieties

The table below reports the mean training macro F1, mean validation macro F1, validation standard deviation and rank for every candidate. A large train-validation gap is evidence of overfitting.

This comparison is important for the final evaluation because it demonstrates that the Decision Tree was tested with multiple preprocessing and hyperparameter varieties rather than only one default model.


In [ ]:
results = pd.DataFrame(search.cv_results_)

def prep_name(prep):
    if prep == "passthrough":
        return "All 16 features"
    return "Mutual information: best 12"

comparison = results[[
    "param_prep",
    "param_tree__criterion",
    "param_tree__max_depth",
    "param_tree__min_samples_leaf",
    "param_tree__ccp_alpha",
    "mean_train_score",
    "mean_test_score",
    "std_test_score",
    "rank_test_score",
]].copy()

comparison.insert(0, "preprocessing", comparison["param_prep"].map(prep_name))
comparison = comparison.drop(columns="param_prep")
comparison = comparison.sort_values(["rank_test_score", "mean_test_score"], ascending=[True, False])

display(comparison.head(20))


## 9. Final evaluation on held-out test data

After GridSearchCV selects the best configuration using only training-fold validation scores, `best_estimator_` is evaluated once on the held-out test set.

The reported metrics are:
- **Accuracy:** overall percentage of correctly classified letters.
- **Macro F1:** gives every letter equal importance.
- **Weighted F1:** weights each letter's F1 by its number of test samples.
- **Classification report:** precision, recall and F1 for each A–Z class.


In [ ]:
best_model = search.best_estimator_
pred = best_model.predict(X_test)

test_accuracy = accuracy_score(y_test, pred)
test_macro_f1 = f1_score(y_test, pred, average="macro")
test_weighted_f1 = f1_score(y_test, pred, average="weighted")

print("Test accuracy:", test_accuracy)
print("Test macro F1:", test_macro_f1)
print("Test weighted F1:", test_weighted_f1)

print(classification_report(
    y_test,
    pred,
    labels=np.arange(26),
    target_names=encoder.classes_,
    zero_division=0,
))


## 10. Confusion matrix and error analysis

Rows are actual letters and columns are predicted letters. A strong diagonal indicates correct predictions. Off-diagonal cells show letter pairs the Decision Tree confuses.

The second cell lists the ten largest specific misclassifications so they can be discussed during the viva.


In [ ]:
fig, ax = plt.subplots(figsize=(14, 14))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    pred,
    labels=np.arange(26),
    display_labels=encoder.classes_,
    ax=ax,
    xticks_rotation=90,
    colorbar=False,
)

ax.set_title("Decision Tree Confusion Matrix")
plt.tight_layout()
plt.show()


In [ ]:
confusions = pd.crosstab(
    pd.Series(encoder.inverse_transform(y_test), name="actual"),
    pd.Series(encoder.inverse_transform(pred), name="predicted"),
)

confusions = confusions.reindex(
    index=encoder.classes_,
    columns=encoder.classes_,
    fill_value=0,
)

for letter in encoder.classes_:
    confusions.loc[letter, letter] = 0

top_confusions = (
    confusions.stack()
    .sort_values(ascending=False)
    .head(10)
    .rename("count")
    .reset_index()
)

display(top_confusions)


## 11. Feature importance

A Decision Tree provides feature importance based on how much each feature contributes to reducing impurity. The chart shows which measurements were most useful to the final tree.

If feature selection is the winning preprocessing branch, importance is mapped back to the selected original feature names.


In [ ]:
final_tree = best_model.named_steps["tree"]
final_prep = best_model.named_steps["prep"]

if final_prep == "passthrough":
    used_features = np.array(X.columns)
else:
    used_features = np.array(X.columns)[final_prep.get_support()]

importance_df = pd.DataFrame({
    "feature": used_features,
    "importance": final_tree.feature_importances_,
}).sort_values("importance", ascending=False)

display(importance_df)

plt.figure(figsize=(10, 6))
plt.barh(
    importance_df["feature"][::-1],
    importance_df["importance"][::-1],
)
plt.xlabel("Feature importance")
plt.title("Decision Tree Feature Importance")
plt.tight_layout()
plt.show()


## 12. Visualize the top of the final tree

The full trained tree may contain many nodes, so plotting the entire tree can be unreadable. For demonstration, this plot shows only the **first three levels** of the final fitted tree while preserving the real learned split rules at those levels.


In [ ]:
plt.figure(figsize=(24, 12))
plot_tree(
    final_tree,
    feature_names=list(used_features),
    class_names=list(encoder.classes_),
    filled=True,
    rounded=True,
    max_depth=3,
    fontsize=7,
)
plt.title("Top Levels of the Best Decision Tree")
plt.tight_layout()
plt.show()

print("Final tree depth:", final_tree.get_depth())
print("Final tree leaves:", final_tree.get_n_leaves())


## 13. Save model and results

This cell saves the fitted best pipeline and evaluation tables so they can be demonstrated or included in the group comparison later.


In [ ]:
project_root = next(
    (p for p in (Path.cwd(), Path.cwd().parent) if (p / "data" / "raw").exists()),
    Path.cwd(),
)

output_dir = project_root / "results" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

comparison.to_csv(output_dir / "decision_tree_cv_comparison.csv", index=False)
importance_df.to_csv(output_dir / "decision_tree_feature_importance.csv", index=False)

summary = pd.DataFrame([{
    "baseline_cv_macro_f1": baseline_scores.mean(),
    "best_cv_macro_f1": search.best_score_,
    "test_accuracy": test_accuracy,
    "test_macro_f1": test_macro_f1,
    "test_weighted_f1": test_weighted_f1,
    "tree_depth": final_tree.get_depth(),
    "tree_leaves": final_tree.get_n_leaves(),
    "train_rows": len(y_train),
    "test_rows": len(y_test),
    "cv_folds": cv.get_n_splits(),
}])
summary.to_csv(output_dir / "decision_tree_summary.csv", index=False)

joblib.dump(best_model, output_dir / "decision_tree_best_model.joblib")
joblib.dump(encoder, output_dir / "decision_tree_label_encoder.joblib")

print("Saved outputs to:", output_dir)
display(summary)


## 14. Conclusion, limitations and viva-ready findings

The cell below generates the conclusion from the **actual current run**, so the numbers will automatically match your executed notebook.

### Limitations
A single Decision Tree can overfit noisy details, especially when it grows very deep. Its predictions can also change if the training data changes. The UCI dataset contains extracted numerical measurements rather than raw images, so this experiment does not prove performance on arbitrary photographs or handwriting.

### Possible improvements
Further work could test a larger pruning range, repeated stratified cross-validation, class-specific error analysis, or ensemble tree methods such as Random Forest / Gradient Boosting. For the individual evaluation, however, this notebook keeps the implemented model family specifically as a **Decision Tree**.


In [ ]:
from IPython.display import Markdown

winner = results.loc[search.best_index_]
branch = prep_name(search.best_params_["prep"])
cv_gain = search.best_score_ - baseline_scores.mean()
train_gap = winner["mean_train_score"] - winner["mean_test_score"]

leading_errors = ", ".join(
    f"{row.actual} → {row.predicted} ({row.count})"
    for row in top_confusions.head(3).itertuples(index=False)
)

display(Markdown(f"""
### Current-run findings

- **Model:** Decision Tree Classifier
- **Candidates trained:** {len(results)}
- **Validation:** {cv.get_n_splits()}-fold Stratified Cross-Validation
- **Selection metric:** Macro F1
- **Baseline CV macro F1:** {baseline_scores.mean():.4f}
- **Best CV macro F1:** {search.best_score_:.4f}
- **CV improvement over baseline:** {cv_gain:+.4f}
- **Winning preprocessing:** {branch}
- **Criterion:** {search.best_params_['tree__criterion']}
- **Max depth setting:** {search.best_params_['tree__max_depth']}
- **Minimum samples per leaf:** {search.best_params_['tree__min_samples_leaf']}
- **ccp_alpha:** {search.best_params_['tree__ccp_alpha']}
- **Train-validation F1 gap:** {train_gap:.4f}
- **Test accuracy:** {test_accuracy:.4f}
- **Test macro F1:** {test_macro_f1:.4f}
- **Test weighted F1:** {test_weighted_f1:.4f}
- **Final fitted tree depth:** {final_tree.get_depth()}
- **Final number of leaves:** {final_tree.get_n_leaves()}
- **Largest observed confusions:** {leading_errors}

**Conclusion:** The tuned Decision Tree should be preferred over the baseline only if its cross-validation performance and generalization are better. The held-out metrics above provide the final estimate for this implementation, while the train-validation gap helps identify possible overfitting.
"""))
